### Data Reading - PySpark

In [0]:
dbutils.fs.ls('/Volumes/workspace/default/big_mart_sales')

In [0]:
df = spark.read.table('workspace.default.big_mart_sales')

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
### DDL Schema
my_ddl_schema='''
                Item_Identifier STRING,
                Item_Weight STRING,
                Item_Fat_Content STRING, 
                Item_Visibility DOUBLE,
                Item_Type STRING,
                Item_MRP DOUBLE,
                Outlet_Identifier STRING,
                Outlet_Establishment_Year INT,
                Outlet_Size STRING,
                Outlet_Location_Type STRING, 
                Outlet_Type STRING,
                Item_Outlet_Sales DOUBLE 

'''

In [0]:
df = spark.read.table('workspace.default.big_mart_sales') 

In [0]:
df.display()

In [0]:
## StruckType Schema
from pyspark.sql.types import * 
from pyspark.sql.functions import *  

In [0]:
df.printSchema()

### Transformations

In [0]:
df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

### Alias

In [0]:
df.select(col('Item_Identifier').alias('Item_Id')).display()

In [0]:
### Filter
## s1
df.filter(col('Item_Fat_Content')=='Regular').display()

In [0]:
### s2
df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).display()

In [0]:
### s3
df.filter((col('Outlet_Size').isNull()) & (col("Outlet_Location_Type").isin('Tier 1', 'Tier 2'))).display()

### WithColumn Renamed

In [0]:
df.withColumnRenamed('Item_Weight','Item_Wt').display()

In [0]:
### with column
df=df.withColumn('flag', lit('new'))
df.display()

In [0]:
df= df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),"Regular", "Reg"))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'),"Low Fat", "LF"))

In [0]:
df.display()

Type Casting

In [0]:
df=df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))

In [0]:
df.printSchema()

Sorting in Pyspark

In [0]:
## s1
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(col('Item_Visibility').asc()).display()
#s2

In [0]:
#s3
df.sort(['Item_Weight','Item_Visibility'], ascending=[0,0]).display()

In [0]:
#s4
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0,1]).display()
#s5
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[1,0]).display()
#s6


In [0]:
#### Limit
df.limit(10).display()

In [0]:
# drop 
df.drop('Item_Visibility').display()

In [0]:
df.drop('Item_Visibility', 'Item_Type').display()

In [0]:
# Drop Duplicates
df.dropDuplicates().display()

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()

In [0]:
df.distinct().display()

Union and Union BY Name

In [0]:
data1 =[('1','kid'),
       ('2','sid') ]
schema1='id STRING, NAME STRING'
df1=spark.createDataFrame(data1,schema1)

data2=[('3','Rahul'),
       ('4','Jas')]
schema2='id STRING, NAME STRING'
df2=spark.createDataFrame(data2,schema2)


In [0]:
df1.display()
df2.display()

In [0]:
df1.union(df2).display()

In [0]:
data1= [('kid','1'),
       ('sid','2') ]
schema1='name STRING, id STRING'
df1=spark.createDataFrame(data1,schema1)

df1.display()

In [0]:
df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

String Functions

In [0]:
# InitCap()
df.select(upper('Item_Type').alias('Upper_Item_Name')).display()

Date Functions

In [0]:
## current Date
df=df.withColumn('curr_date', current_date())
df.display()

In [0]:
# Date_Add()
df=df.withColumn('week_after', date_add('curr_date',7))

In [0]:
df.display()

In [0]:
#date_sub()
df.withColumn('Week_before', date_sub('curr_date',7)).display()

In [0]:
df=df.withColumn('Week_before', date_sub('curr_date',-7))
df.display()

In [0]:
#date Diff
df=df.withColumn('datediff',datediff('Week_before', 'curr_date'))
df.display()

In [0]:
#date format
df=df.withColumn('Week_before', date_format('Week_before','dd-MM-yyyy'))
df.display()

Handling the Null Values

In [0]:
df.dropna('all').display()

In [0]:
df.dropna('any').display()

In [0]:
df.dropna(subset=['Outlet_Size']).display()

In [0]:
df.display()

In [0]:
#Filling Nulls
df.fillna('NotAvailable').display()

In [0]:
df.fillna('NotAvailable', subset=['Outlet_Size']).display()

Split and Indexing

In [0]:
#split
df.withColumn('OutLet_Type',split('Outlet_Type',' ')).display()

In [0]:
df.display()

In [0]:
#indexing
df.withColumn('Outlet__Type',split('Outlet_Type',' ')[1]).display()

Explode

In [0]:
df_exp=df.withColumn('Outlet_Type',split('Outlet_Type',' '))
df_exp.display()

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

In [0]:
df_exp.withColumn('Type1_falg', array_contains('Outlet_Type','Type1')).display()

Group By

In [0]:
#s1
#df.display()
df.groupBy('Item_Type').agg(sum('Item_MRP')).display()

In [0]:
#s2
df.groupBy('Item_Type').agg(avg('Item_MRP')).display()

In [0]:
#s3
df.groupBy('Item_Type', 'Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).display()

In [0]:
#s4
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'), avg('Item_MRP')).display()

Collect List

In [0]:
data=[
    ('user1','book1'),
    ('user1', 'book2'),
    ('user2', 'book2'),
    ('user2', 'book4'),
    ('user3', 'book1')
]
schema='user String, Book String'
df_book=spark.createDataFrame(data,schema)
df_book.display()

In [0]:
df_book.groupBy('User').agg(collect_list('book')).display()

In [0]:
df.select('Item_Type','Outlet_Size','Item_MRP').display()

In [0]:
#pivot
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

When Otherwise

In [0]:
#s1
df=df.withColumn('veg_flag', when(col('Item_Type')=='Meat','Non-Veg').otherwise('veg'))
df.display()

In [0]:
df.withColumn('veg_exp_flag',when(((col('veg_flag')=='veg') & (col('Item_MRP')<100)), 'veg_Inexpensive')\
    .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'veg_Expensive')
    .otherwise('Non_Veg')).display()

Joins

In [0]:
dataj1 = [('1','gaur','d01'),
          ('2','kit','d02'),
          ('3','sam','d03'),
          ('4','tim','d03'),
          ('5','aman','d05'),
          ('6','nad','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

In [0]:
df1.display()

In [0]:
df2.display()

In [0]:
# Inner Join
df1.join(df2,df1['dept_id']==df2['dept_id'],'inner').display()
#left Join
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').display()
#Right Join
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').display()
#Anti Join
df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').display()


Window Functions

In [0]:
#Row Number()
#df.display()
from pyspark.sql.window import Window
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).display()

In [0]:
# row vs dense Rank 
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('dense_rank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

In [0]:
df.withColumn('dum',sum('Item_MRP').over(Window.orderBy('Item_Identifier').rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

User Defined Functions

In [0]:
#step1
def my_fun(x):
    return x*x;

In [0]:
#step 2
my_udf=udf(my_fun)

In [0]:
df.withColumn('mynewcol',my_udf('Item_MRP')).display()

Data Writing

In [0]:
%sql
show VOLUMES IN workspace.default

In [0]:
#csv
df.write.format('csv')\
    .option('header', True)\
    .mode('overwrite').save('/Volumes/workspace/default/big_mart_sales/data.csv')

append

In [0]:
df.write.format('CSV')\
    .mode('append')\
        .save('/Volumes/workspace/default/big_mart_sales/data.csv')

In [0]:
df.write.format('csv')\
    .mode('append')\
    .option('path', '/Volumes/workspace/default/big_mart_sales/data.csv')\
    .save()

In [0]:
#overwrite
df.write.format('CSV')\
    .mode('overwrite')\
        .option('path', '/Volumes/workspace/default/big_mart_sales/data.csv')\
    .save()

In [0]:
#ignore
df.write.format('csv')\
    .mode('ignore')\
    .option('path', '/Volumes/workspace/default/big_mart_sales/data.csv')\
    .save()

In [0]:
#parquet
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path', '/Volumes/workspace/default/big_mart_sales/data.parquet')\
    .save()
#

In [0]:
#table
df.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('workspace.default.mytable')
#overwrite
df.write.format('parquet')

Spark SQL

In [0]:
# Create temp View
df.createTempView('my_view')

In [0]:
%sql
select * from my_view where Item_Fat_Content='Lf'

In [0]:
df_sql=spark.sql("select * from my_view where Item_Fat_Content='Lf'")

In [0]:
df_sql.display()

Thank You